# Chapter 11: Native Multimodal & Any-to-Any Models

## From "Glue" Architectures to Unified Token Spaces

---

**Papers covered:**
- **Chameleon** (Meta, 2024) — Early-fusion autoregressive over all modalities
- **Transfusion** (Meta, 2024) — Autoregressive text + diffusion images in one model
- **Emu3** (BAAI, 2024) — Next-token prediction is all you need (for vision too)
- **Janus Pro** (DeepSeek, 2025) — Decoupled visual encoding for understanding vs. generation

**What you will learn:**
1. How VQ-VAE tokenizers convert continuous signals (images, audio) into discrete token sequences
2. How to build unified vocabularies spanning text, image, and audio modalities
3. The architectural tradeoffs: pure autoregressive vs. transfusion vs. decoupled encoders
4. Why generation and understanding may need fundamentally different representations
5. How to implement a simplified any-to-any generation pipeline

**Prerequisites:** Transformers, backpropagation, basic familiarity with CNNs and language modeling.
No prior knowledge of VQ-VAE or multimodal generation is assumed.

---

### The Paradigm Shift

```
"Glue" models (LLaVA, Flamingo):           Native multimodal (this chapter):
  Image → [frozen encoder] → adapt → LLM     Image → [tokenizer] → discrete tokens ─┐
  Text  → [tokenizer] → LLM                  Text  → [tokenizer] → discrete tokens ──┤
                                              Audio → [tokenizer] → discrete tokens ──┤
  Can only UNDERSTAND images                                                           ▼
  Can only GENERATE text                      ONE model, ONE vocabulary, ONE loss
                                              Can UNDERSTAND and GENERATE everything
```

The core insight is deceptively simple: **if you can convert any signal into discrete tokens, you can train a single autoregressive model over all of them.** The difficulty lies in building tokenizers that preserve enough information for faithful reconstruction while being compact enough for tractable sequence lengths.

# 1) The "Token Everything" Paradigm

## 1.1 Why Discretization?

Language models operate over **discrete vocabularies.** Each token is an integer index into an embedding table. This discreteness is what enables:

- **Unified cross-entropy training** — one loss function for all modalities
- **Autoregressive generation** — sample token by token from a categorical distribution
- **Shared attention** — text tokens can attend to image tokens and vice versa

But images, audio, and video are **continuous signals.** The bridge is **vector quantization (VQ):** learn a finite codebook of prototype vectors, then represent any continuous vector by the index of its nearest codebook entry.

## 1.2 The Tokenization Pipeline

```
┌─────────────┐    ┌──────────┐    ┌───────────┐    ┌──────────────┐
│ Raw Signal  │───▶│ Encoder  │───▶│ Quantizer │───▶│ Token IDs    │
│ (256×256×3) │    │ (CNN/ViT)│    │ (VQ)      │    │ (16×16 = 256)│
└─────────────┘    └──────────┘    └───────────┘    └──────────────┘
                                                           │
                                                           ▼
                                                    ┌──────────────┐
                                                    │ Decoder      │
                                                    │ (CNN/ViT)    │
                                                    └──────┬───────┘
                                                           │
                                                           ▼
                                                    ┌──────────────┐
                                                    │ Reconstructed│
                                                    │ (256×256×3)  │
                                                    └──────────────┘
```

**Key design parameters:**
| Parameter | Typical Range | Effect |
|-----------|--------------|--------|
| Codebook size | 1K – 16K | Larger → finer detail, but harder to train |
| Spatial downsampling | 8× – 32× | More compression → shorter sequences, but loses detail |
| Embedding dimension | 128 – 512 | Capacity of each codebook vector |
| Number of codebooks (RQ/FSQ) | 1 – 16 | Multiple passes for residual refinement |

### Sample Input/Output for Image Tokenization

```
Input:  RGB image tensor of shape (1, 3, 256, 256)  — a photo of a cat
Output: Token sequence of shape (1, 256) — 256 integers in [0, 8191]
        e.g., [4021, 7193, 301, 5544, ..., 1082]  (each = codebook index)

Reconstruction: Decode tokens back → (1, 3, 256, 256) — should resemble original cat
```

# 2) Environment Setup

In [ ]:
!pip install torch torchvision matplotlib numpy einops pillow tqdm -q

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import matplotlib.pyplot as plt
import numpy as np
from einops import rearrange, repeat
from tqdm.auto import tqdm
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict
import math
import warnings

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

# 3) VQ-VAE Image Tokenizer — From Scratch

## 3.1 Motivation: Why VQ-VAE?

The **Vector Quantized Variational Autoencoder** (VQ-VAE, van den Oord et al., 2017) solves a specific problem: how to represent images as sequences of discrete tokens while maintaining reconstruction quality.

**Why not just use raw pixel values?**
A 256×256 RGB image has 196,608 values. Even with an 8-bit vocabulary (256 possible values per pixel), the sequence length is intractable for attention (O(n²) cost). VQ-VAE compresses this to ~256 tokens by:

1. **Spatial compression** via a convolutional encoder (256×256 → 16×16 spatial)
2. **Discretization** via vector quantization (continuous → codebook indices)

**Why not a standard VAE with continuous latents?**
Continuous latents require a separate generation model (e.g., diffusion). Discrete tokens let us use the **same autoregressive LM** for both text and images.

### Sample Input/Output

```
Input to VQ-VAE:
  x = torch.randn(4, 3, 128, 128)          # batch of 4 RGB images

Output from VQ-VAE:
  codes     = tensor of shape (4, 16, 16)   # 256 discrete codes per image, each in [0, codebook_size)
  x_recon   = tensor of shape (4, 3, 128, 128)  # reconstructed images
  vq_loss   = scalar                        # commitment + codebook loss
```

## 3.2 The Encoder

The encoder maps an image to a spatial grid of continuous feature vectors. We use a standard convolutional architecture with residual blocks and progressive downsampling.

In [ ]:
class ResidualBlock(nn.Module):
    """
    Pre-activation residual block with GroupNorm.
    Maintains spatial dimensions; only transforms channel features.

    Why GroupNorm over BatchNorm?
    - GroupNorm is independent of batch size, critical for small-batch
      training common in multimodal settings.
    - More stable gradients during VQ training where the loss landscape
      has sharp discontinuities from the argmin operation.
    """

    def __init__(self, channels: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.GroupNorm(num_groups=32, num_channels=channels),
            nn.SiLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.GroupNorm(num_groups=32, num_channels=channels),
            nn.SiLU(),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, channels, height, width) → (batch_num, channels, height, width)
        return x + self.block(x)


class Downsample(nn.Module):
    """
    Strided convolution for 2× spatial downsampling.
    Preferred over pooling because it's learnable — the network can
    choose what information to preserve during compression.
    """

    def __init__(self, channels: int):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, stride=2, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, channels, H, W) → (batch_num, channels, H/2, W/2)
        return self.conv(x)


class Upsample(nn.Module):
    """
    Nearest-neighbor upsampling + convolution for 2× spatial increase.
    Avoids checkerboard artifacts from transposed convolutions.
    """

    def __init__(self, channels: int):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, kernel_size=3, padding=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, channels, H, W) → (batch_num, channels, 2H, 2W)
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        return self.conv(x)


class Encoder(nn.Module):
    """
    Image encoder: progressively downsamples spatial dimensions while
    increasing channel capacity.

    Architecture:
      (batch_num, 3, H, W)
        → initial conv  → (batch_num, base_ch, H, W)
        → [ResBlock + Downsample] × num_downsamples
        → (batch_num, base_ch * 2^n, H/2^n, W/2^n)
        → final proj   → (batch_num, embed_dim, H/2^n, W/2^n)

    For H=W=128, num_downsamples=3: output is (batch_num, embed_dim, 16, 16)
    """

    def __init__(
        self,
        in_channels: int = 3,
        base_channels: int = 128,
        channel_multipliers: Tuple[int, ...] = (1, 2, 2, 4),
        num_res_blocks: int = 2,
        embed_dim: int = 256,
    ):
        super().__init__()
        channels = base_channels

        # Initial projection from RGB to feature space
        # (batch_num, 3, H, W) → (batch_num, base_channels, H, W)
        layers = [nn.Conv2d(in_channels, channels, kernel_size=3, padding=1)]

        # Progressive downsampling stages
        for i, mult in enumerate(channel_multipliers):
            out_channels = base_channels * mult

            # Residual blocks at this resolution
            for _ in range(num_res_blocks):
                layers.append(ResidualBlock(channels))
                if channels != out_channels:
                    layers.append(nn.Conv2d(channels, out_channels, kernel_size=1))
                    channels = out_channels

            # Downsample (except at the last stage)
            if i < len(channel_multipliers) - 1:
                layers.append(Downsample(channels))

        # Final residual processing at lowest resolution
        layers.extend([ResidualBlock(channels), ResidualBlock(channels)])

        # Project to embedding dimension for codebook lookup
        # (batch_num, channels, h, w) → (batch_num, embed_dim, h, w)
        layers.extend(
            [
                nn.GroupNorm(num_groups=32, num_channels=channels),
                nn.SiLU(),
                nn.Conv2d(channels, embed_dim, kernel_size=1),
            ]
        )

        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, 3, H, W) → (batch_num, embed_dim, h, w)
        return self.net(x)


print("Encoder architecture defined.")
print(f"For 128×128 input with 3 downsamples: output spatial = 16×16")
print(f"Sequence length for LLM = 16 × 16 = 256 tokens per image")

## 3.3 The Vector Quantization Layer — Core of the Tokenizer

This is the heart of VQ-VAE and the key enabler of "images as language."

### The Quantization Process

```
Encoder output z_e:  (batch_num, embed_dim, h, w)
                          ↓
Reshape to vectors:  (batch_num × h × w, embed_dim) — N vectors, each embed_dim-dimensional
                          ↓
For each vector, find nearest codebook entry:
    code_idx = argmin_k ||z_e - e_k||²    where e_k are codebook vectors
                          ↓
Replace with codebook entry:
    z_q = e_{code_idx}   — the quantized representation
                          ↓
Reshape back:        (batch_num, embed_dim, h, w) — same shape, but now discrete
```

### The Gradient Problem

`argmin` has **zero gradients everywhere** (piecewise constant). VQ-VAE uses the **straight-through estimator:** during backprop, copy gradients from decoder input directly to encoder output, bypassing the quantization step.

```
Forward:  z_q = codebook_lookup(argmin(distances))     ← discrete
Backward: ∂L/∂z_e = ∂L/∂z_q                           ← straight-through (pretend quantization didn't happen)
```

### Three-Part Loss

```
L = L_reconstruction + β·L_commitment + L_codebook

L_reconstruction = ||x - decoder(z_q)||²     — reconstruct the image
L_commitment     = ||z_e - sg(z_q)||²         — push encoder towards codebook (sg = stop gradient)
L_codebook       = ||sg(z_e) - z_q||²         — push codebook towards encoder outputs
```

The commitment loss prevents the encoder from "running away" — producing embeddings far from any codebook entry. Without it, the encoder output space drifts and codebook utilization collapses.

In [ ]:
class VectorQuantizer(nn.Module):
    """
    Vector Quantization layer with EMA codebook updates and codebook reset.

    Design decisions:
    1. EMA updates (vs. gradient-based): More stable training, avoids codebook
       collapse where many entries go unused. Used in all major VQ-VAE papers.
    2. Codebook reset: Re-initialize dead codes by replacing them with encoder
       outputs. Critical for maintaining high codebook utilization.
    3. L2-normalized lookup: Optional, used in some recent works (Emu3) to
       stabilize the distance computation in high dimensions.
    """

    def __init__(
        self,
        codebook_size: int = 8192,
        embed_dim: int = 256,
        commitment_weight: float = 0.25,
        decay: float = 0.99,
        eps: float = 1e-5,
        reset_threshold: int = 2,
    ):
        super().__init__()
        self.codebook_size = codebook_size
        self.embed_dim = embed_dim
        self.commitment_weight = commitment_weight
        self.decay = decay
        self.eps = eps
        self.reset_threshold = reset_threshold

        # Codebook embeddings — initialized from N(0, 1)
        # Not an nn.Parameter because we update via EMA, not gradients
        embedding = torch.randn(codebook_size, embed_dim)
        self.register_buffer("embedding", embedding)

        # EMA tracking: running count of how often each code is used,
        # and the running sum of encoder vectors assigned to each code
        self.register_buffer("ema_count", torch.zeros(codebook_size))
        self.register_buffer("ema_weight", embedding.clone())

        # Track usage for codebook reset
        self.register_buffer("usage_count", torch.zeros(codebook_size))

    def _find_nearest(self, z_flat: torch.Tensor) -> torch.Tensor:
        """
        Find nearest codebook entry for each input vector using
        efficient expanded-form L2 distance.

        ||z - e||² = ||z||² + ||e||² - 2·z·eᵀ

        This avoids materializing a (N, codebook_size, embed_dim) tensor.
        """
        # (num_tokens, embed_dim) → (num_tokens,)
        z_sq = (z_flat**2).sum(dim=1, keepdim=True)

        # (codebook_size, embed_dim) → (codebook_size,)
        e_sq = (self.embedding**2).sum(dim=1, keepdim=True).t()

        # (num_tokens, embed_dim) × (embed_dim, codebook_size) → (num_tokens, codebook_size)
        dot = z_flat @ self.embedding.t()

        # (num_tokens, codebook_size)
        distances = z_sq + e_sq - 2 * dot

        # (num_tokens,) — index of nearest codebook vector
        return distances.argmin(dim=1)

    def _ema_update(self, z_flat: torch.Tensor, indices: torch.Tensor):
        """
        Exponential Moving Average codebook update (Appendix A of VQ-VAE paper).

        Instead of backpropagating gradients through the codebook,
        we maintain running averages:
          N_k ← γ·N_k + (1-γ)·n_k           (count of vectors assigned to code k)
          m_k ← γ·m_k + (1-γ)·Σ{z: code(z)=k}   (sum of assigned vectors)
          e_k ← m_k / N_k                    (updated codebook entry)

        Why EMA? Gradient-based updates to the codebook via the VQ loss often
        cause codebook collapse: a few codes dominate, most go unused.
        EMA gives each code a "memory" proportional to its usage frequency.
        """
        # One-hot encoding of assignments
        # (num_tokens,) → (num_tokens, codebook_size)
        onehot = F.one_hot(indices, self.codebook_size).float()

        # Count how many vectors were assigned to each code in this batch
        # (codebook_size,)
        batch_count = onehot.sum(dim=0)

        # Sum of encoder outputs assigned to each code
        # (codebook_size, embed_dim)
        batch_sum = onehot.t() @ z_flat

        # EMA update for counts
        self.ema_count.mul_(self.decay).add_(batch_count, alpha=1 - self.decay)

        # EMA update for embedding sums
        self.ema_weight.mul_(self.decay).add_(batch_sum, alpha=1 - self.decay)

        # Laplace smoothing to prevent division by zero
        n = self.ema_count.sum()
        count_smoothed = (
            (self.ema_count + self.eps) / (n + self.codebook_size * self.eps) * n
        )

        # Update codebook entries
        self.embedding.copy_(self.ema_weight / count_smoothed.unsqueeze(1))

        # Track usage for dead code reset
        self.usage_count.add_(batch_count)

    def _reset_dead_codes(self, z_flat: torch.Tensor):
        """
        Replace dead codebook entries (unused codes) with randomly
        sampled encoder outputs.

        Why this matters:
        Without reset, codebook utilization often drops to 10-30% of entries.
        Dead codes represent wasted capacity. In Emu3 and Chameleon, high
        codebook utilization (>95%) is critical for reconstruction quality.
        """
        dead_mask = self.usage_count < self.reset_threshold
        num_dead = dead_mask.sum().item()

        if num_dead > 0 and z_flat.shape[0] > 0:
            # Randomly sample encoder outputs to replace dead codes
            replace_indices = torch.randint(0, z_flat.shape[0], (num_dead,))
            self.embedding[dead_mask] = z_flat[replace_indices].detach()
            self.ema_weight[dead_mask] = z_flat[replace_indices].detach()
            self.ema_count[dead_mask] = 1.0

        # Reset usage tracking periodically
        self.usage_count.zero_()

    def forward(
        self, z_e: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Quantize encoder output to nearest codebook entries.

        Args:
            z_e: Encoder output, (batch_num, embed_dim, h, w)

        Returns:
            z_q:     Quantized output, (batch_num, embed_dim, h, w)
            indices: Codebook indices, (batch_num, h, w)
            vq_loss: Combined commitment + codebook loss (scalar)
        """
        batch_num, embed_dim, h, w = z_e.shape

        # Reshape: (batch_num, embed_dim, h, w) → (batch_num*h*w, embed_dim)
        z_flat = rearrange(z_e, "b d h w -> (b h w) d")

        # Find nearest codebook entry for each spatial position
        # (num_tokens,) where num_tokens = batch_num * h * w
        indices = self._find_nearest(z_flat)

        # Lookup quantized vectors
        # (num_tokens,) → (num_tokens, embed_dim)
        z_q_flat = self.embedding[indices]

        # EMA codebook update (only during training)
        if self.training:
            self._ema_update(z_flat, indices)

        # Commitment loss: push encoder output towards codebook
        # (scalar) — MSE between encoder output and its assigned codebook vector
        commitment_loss = F.mse_loss(z_flat.detach(), z_q_flat)
        encoder_loss = F.mse_loss(z_flat, z_q_flat.detach())
        vq_loss = commitment_loss + self.commitment_weight * encoder_loss

        # Straight-through estimator: copy gradients from z_q to z_e
        # Forward: z_q (discrete), Backward: gradients flow to z_e
        z_q_flat = z_flat + (z_q_flat - z_flat).detach()

        # Reshape back to spatial
        # (num_tokens, embed_dim) → (batch_num, embed_dim, h, w)
        z_q = rearrange(z_q_flat, "(b h w) d -> b d h w", b=batch_num, h=h, w=w)

        # (num_tokens,) → (batch_num, h, w)
        indices = rearrange(indices, "(b h w) -> b h w", b=batch_num, h=h, w=w)

        return z_q, indices, vq_loss

    def codes_to_embeddings(self, codes: torch.Tensor) -> torch.Tensor:
        """
        Convert codebook indices back to embedding vectors (for decoding).

        Args:
            codes: (batch_num, h, w) integer tensor

        Returns:
            z_q: (batch_num, embed_dim, h, w) float tensor
        """
        batch_num, h, w = codes.shape

        # (batch_num*h*w,) → (batch_num*h*w, embed_dim)
        z_q_flat = self.embedding[codes.reshape(-1)]

        # (batch_num*h*w, embed_dim) → (batch_num, embed_dim, h, w)
        return rearrange(z_q_flat, "(b h w) d -> b d h w", b=batch_num, h=h, w=w)

    @property
    def utilization(self) -> float:
        """Fraction of codebook entries that have been used."""
        return (self.usage_count > 0).float().mean().item()


# Quick sanity check
vq = VectorQuantizer(codebook_size=512, embed_dim=256)
dummy_z = torch.randn(2, 256, 16, 16)
z_q, indices, loss = vq(dummy_z)
print(f"Input shape:  {dummy_z.shape}")
print(f"Output shape: {z_q.shape}")
print(f"Codes shape:  {indices.shape}")
print(f"Code range:   [{indices.min().item()}, {indices.max().item()}]")
print(f"VQ loss:      {loss.item():.4f}")

## 3.4 The Decoder

The decoder mirrors the encoder: it takes the quantized feature map and progressively upsamples it back to image resolution. The decoder's quality directly determines the **reconstruction ceiling** — no matter how good the LLM is at predicting image tokens, the final image quality is bounded by the decoder's ability to reconstruct from those tokens.

This is why Chameleon and Emu3 invest heavily in tokenizer quality: a poor tokenizer creates an information bottleneck that no amount of LLM scaling can overcome.

In [ ]:
class Decoder(nn.Module):
    """
    Image decoder: upsamples quantized features back to pixel space.

    Architecture mirrors the encoder in reverse:
      (batch_num, embed_dim, h, w)
        → initial proj  → (batch_num, max_ch, h, w)
        → [ResBlock + Upsample] × num_upsamples
        → (batch_num, base_ch, H, W)
        → final conv    → (batch_num, 3, H, W)
    """

    def __init__(
        self,
        out_channels: int = 3,
        base_channels: int = 128,
        channel_multipliers: Tuple[int, ...] = (1, 2, 2, 4),
        num_res_blocks: int = 2,
        embed_dim: int = 256,
    ):
        super().__init__()

        # Start from the deepest (most compressed) representation
        channels = base_channels * channel_multipliers[-1]

        # Project from embedding space to decoder feature space
        # (batch_num, embed_dim, h, w) → (batch_num, channels, h, w)
        layers = [
            nn.Conv2d(embed_dim, channels, kernel_size=3, padding=1),
            ResidualBlock(channels),
            ResidualBlock(channels),
        ]

        # Progressive upsampling stages (reverse order of encoder)
        reversed_mults = list(reversed(channel_multipliers))
        for i, mult in enumerate(reversed_mults):
            out_channels_stage = base_channels * mult

            for _ in range(num_res_blocks):
                layers.append(ResidualBlock(channels))
                if channels != out_channels_stage:
                    layers.append(nn.Conv2d(channels, out_channels_stage, kernel_size=1))
                    channels = out_channels_stage

            # Upsample (except at the last stage)
            if i < len(reversed_mults) - 1:
                layers.append(Upsample(channels))

        # Final projection to RGB
        # (batch_num, base_channels, H, W) → (batch_num, 3, H, W)
        layers.extend(
            [
                nn.GroupNorm(num_groups=32, num_channels=channels),
                nn.SiLU(),
                nn.Conv2d(channels, out_channels, kernel_size=3, padding=1),
            ]
        )

        self.net = nn.Sequential(*layers)

    def forward(self, z_q: torch.Tensor) -> torch.Tensor:
        # (batch_num, embed_dim, h, w) → (batch_num, 3, H, W)
        return self.net(z_q)


print("Decoder defined — mirrors encoder with progressive upsampling.")

## 3.5 Complete VQ-VAE: Putting It Together

The full model chains Encoder → VectorQuantizer → Decoder. During training, we optimize:

$$\mathcal{L} = \underbrace{\|x - \hat{x}\|^2_{2}}_{\text{reconstruction}} + \underbrace{\|\text{sg}[z_e] - z_q\|^2_{2}}_{\text{codebook (EMA)}} + \underbrace{\beta \|z_e - \text{sg}[z_q]\|^2_{2}}_{\text{commitment}}$$

where sg[·] is the stop-gradient operator.

### Sample Input/Output

```
# Training
x = batch of images, shape (batch_num, 3, 128, 128)
x_recon, codes, total_loss = vqvae(x)
# x_recon.shape = (batch_num, 3, 128, 128)   — reconstructed images
# codes.shape   = (batch_num, 16, 16)         — discrete token grid
# total_loss    = scalar                       — reconstruction + VQ losses

# Inference: encode only (for feeding to LLM)
codes = vqvae.encode(x)    # (batch_num, 16, 16)

# Inference: decode only (LLM-generated tokens → image)
x_decoded = vqvae.decode(codes)  # (batch_num, 3, 128, 128)
```

In [ ]:
class VQVAE(nn.Module):
    """
    Complete VQ-VAE for image tokenization.

    Converts images to discrete token sequences (and back).
    This enables 'image as language' — treating image generation
    as a next-token prediction problem.

    Image (128×128×3)
      → Encoder → Latent (16×16×256)
      → Quantize → Codes (16×16) = 256 discrete tokens
      → Decoder → Reconstructed image (128×128×3)
    """

    def __init__(
        self,
        in_channels: int = 3,
        base_channels: int = 128,
        channel_multipliers: Tuple[int, ...] = (1, 2, 2, 4),
        num_res_blocks: int = 2,
        embed_dim: int = 256,
        codebook_size: int = 8192,
        commitment_weight: float = 0.25,
    ):
        super().__init__()
        self.encoder = Encoder(
            in_channels, base_channels, channel_multipliers, num_res_blocks, embed_dim
        )
        self.quantizer = VectorQuantizer(
            codebook_size=codebook_size,
            embed_dim=embed_dim,
            commitment_weight=commitment_weight,
        )
        self.decoder = Decoder(
            in_channels, base_channels, channel_multipliers, num_res_blocks, embed_dim
        )

    def forward(
        self, x: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Full forward pass: encode → quantize → decode.

        Args:
            x: Input images, (batch_num, 3, H, W)
        Returns:
            x_recon: Reconstructed images, (batch_num, 3, H, W)
            codes:   Codebook indices, (batch_num, h, w)
            loss:    Reconstruction + VQ loss (scalar)
        """
        # Encode: (batch_num, 3, H, W) → (batch_num, embed_dim, h, w)
        z_e = self.encoder(x)

        # Quantize: (batch_num, embed_dim, h, w) → (batch_num, embed_dim, h, w) + codes
        z_q, codes, vq_loss = self.quantizer(z_e)

        # Decode: (batch_num, embed_dim, h, w) → (batch_num, 3, H, W)
        x_recon = self.decoder(z_q)

        # Reconstruction loss (L2)
        recon_loss = F.mse_loss(x_recon, x)

        total_loss = recon_loss + vq_loss

        return x_recon, codes, total_loss

    @torch.no_grad()
    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Encode images to discrete token sequences."""
        z_e = self.encoder(x)
        _, codes, _ = self.quantizer(z_e)
        # (batch_num, h, w) — flatten to (batch_num, h*w) for LLM input
        return codes

    @torch.no_grad()
    def decode(self, codes: torch.Tensor) -> torch.Tensor:
        """Decode discrete tokens back to images."""
        z_q = self.quantizer.codes_to_embeddings(codes)
        return self.decoder(z_q)

    @property
    def codebook_utilization(self) -> float:
        return self.quantizer.utilization


# Instantiate and check parameter count
vqvae = VQVAE(
    base_channels=64,
    channel_multipliers=(1, 2, 4),
    num_res_blocks=2,
    embed_dim=256,
    codebook_size=512,
).to(DEVICE)

total_params = sum(p.numel() for p in vqvae.parameters())
print(f"VQ-VAE parameters: {total_params:,}")

# Verify shapes with a dummy input
dummy = torch.randn(2, 3, 128, 128).to(DEVICE)
x_recon, codes, loss = vqvae(dummy)
print(f"Input:   {dummy.shape}")
print(f"Recon:   {x_recon.shape}")
print(f"Codes:   {codes.shape} → {codes.shape[1] * codes.shape[2]} tokens per image")
print(f"Loss:    {loss.item():.4f}")

## 3.6 Training the VQ-VAE on CIFAR-10

We train on CIFAR-10 (32×32 images, resized to 128×128) to demonstrate the full pipeline. In practice, models like Chameleon train their tokenizers on hundreds of millions of images.

Key metrics to watch:
- **Reconstruction loss:** Should decrease steadily — lower means better image quality
- **Codebook utilization:** Should stay high (>80%) — low means many dead codes
- **Perceptual quality:** Visual inspection of reconstructions

In [ ]:
# Load CIFAR-10 and resize to 128×128
transform = T.Compose(
    [
        T.Resize((128, 128)),
        T.ToTensor(),
        # Normalize to [-1, 1] for better gradient dynamics
        T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
    ]
)

train_dataset = torchvision.datasets.CIFAR10(
    root="./data", train=True, download=True, transform=transform
)
train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=64, shuffle=True, num_workers=2, drop_last=True
)

print(f"Training samples: {len(train_dataset):,}")
print(f"Batches per epoch: {len(train_loader)}")

In [ ]:
def train_vqvae(
    model: VQVAE,
    dataloader: torch.utils.data.DataLoader,
    num_epochs: int = 5,
    lr: float = 3e-4,
    device: torch.device = DEVICE,
) -> Dict[str, List[float]]:
    """
    Train VQ-VAE with Adam optimizer and learning rate warmup.

    Returns:
        Dictionary of training metrics per epoch.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # Linear warmup over first epoch, then constant
    warmup_steps = len(dataloader)

    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        return 1.0

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    history = {"loss": [], "utilization": []}

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        num_batches = 0

        pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{num_epochs}")
        for batch_idx, (images, _) in enumerate(pbar):
            # (batch_num, 3, 128, 128)
            images = images.to(device)

            x_recon, codes, loss = model(images)

            optimizer.zero_grad()
            loss.backward()

            # Gradient clipping for stability during VQ training
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

            optimizer.step()
            scheduler.step()

            epoch_loss += loss.item()
            num_batches += 1

            # Reset dead codes periodically
            if batch_idx % 100 == 0 and batch_idx > 0:
                z_e = model.encoder(images)
                z_flat = rearrange(z_e, "b d h w -> (b h w) d")
                model.quantizer._reset_dead_codes(z_flat)

            pbar.set_postfix(
                loss=f"{loss.item():.4f}", util=f"{model.codebook_utilization:.1%}"
            )

        avg_loss = epoch_loss / num_batches
        util = model.codebook_utilization
        history["loss"].append(avg_loss)
        history["utilization"].append(util)
        print(
            f"Epoch {epoch + 1}: loss={avg_loss:.4f}, "
            f"codebook_utilization={util:.1%}"
        )

    return history


# Train for a few epochs (increase for better results)
history = train_vqvae(vqvae, train_loader, num_epochs=3)

In [ ]:
def visualize_reconstructions(
    model: VQVAE,
    dataloader: torch.utils.data.DataLoader,
    num_images: int = 8,
    device: torch.device = DEVICE,
):
    """Show original images alongside their VQ-VAE reconstructions."""
    model.eval()
    images, _ = next(iter(dataloader))
    images = images[:num_images].to(device)

    with torch.no_grad():
        recon, codes, _ = model(images)

    # Denormalize from [-1, 1] to [0, 1]
    images_viz = (images.cpu() * 0.5 + 0.5).clamp(0, 1)
    recon_viz = (recon.cpu() * 0.5 + 0.5).clamp(0, 1)

    fig, axes = plt.subplots(2, num_images, figsize=(2 * num_images, 4))
    for i in range(num_images):
        axes[0, i].imshow(images_viz[i].permute(1, 2, 0))
        axes[0, i].set_title("Original" if i == 0 else "")
        axes[0, i].axis("off")

        axes[1, i].imshow(recon_viz[i].permute(1, 2, 0))
        axes[1, i].set_title("Reconstructed" if i == 0 else "")
        axes[1, i].axis("off")

    plt.suptitle(
        f"VQ-VAE Reconstructions (codebook size={model.quantizer.codebook_size}, "
        f"tokens per image={codes.shape[1]*codes.shape[2]})",
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()

    # Show the discrete token grid for one image
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(codes[0].cpu().numpy(), cmap="tab20", interpolation="nearest")
    ax.set_title(f"Token Grid ({codes.shape[1]}×{codes.shape[2]}) — Each cell = one discrete token")
    ax.set_xlabel("Spatial width")
    ax.set_ylabel("Spatial height")
    plt.colorbar(ax.images[0], ax=ax, label="Codebook index")
    plt.tight_layout()
    plt.show()


visualize_reconstructions(vqvae, train_loader)

# 4) Building a Unified Vocabulary

## 4.1 The Key Idea

In traditional LLMs, the vocabulary is text-only: ~32K–128K BPE tokens. In native multimodal models, we **extend the vocabulary** to include tokens from all modalities:

```
Vocabulary Layout:
┌─────────────────────────────────────────────────────────────┐
│  [0 .. 32000)       Text tokens (BPE/SentencePiece)        │
│  [32000 .. 40192)   Image tokens (VQ-VAE codebook)         │
│  [40192 .. 44288)   Audio tokens (SoundStream codebook)    │
│  [44288 .. 44292)   Special tokens (<IMG>, </IMG>, etc.)   │
└─────────────────────────────────────────────────────────────┘
```

This is the same principle as how multilingual models add language-specific tokens — we're just adding "visual language" and "audio language" to the vocabulary.

### Chameleon's Approach
- 65,536 text tokens (BPE)
- 8,192 image tokens (custom VQ tokenizer trained on 1B images)
- Special tokens: `<image>`, `</image>` to delimit modalities

### Emu3's Approach
- 64,000 text tokens
- 32,768 image tokens (SDXL-based VQ-VAE)
- 4 special tokens for image structure: `<|start_of_image|>`, `<|end_of_image|>`, `<|next_line|>`, `<|image_padding|>`

### Sample: Unified Sequence

```
Text:  "A photo of a sunset over the ocean"
       → [A, photo, of, a, sun, set, over, the, ocean]
       → [65, 2910, 315, 64, 4529, 697, 1065, 279, 18435]

Image: [sunset photo, 256 tokens from VQ-VAE]
       → [32000+4021, 32000+7193, 32000+301, ..., 32000+1082]

Combined: [BOS, 65, 2910, ..., 18435, <IMG>, 36021, 39193, ..., 33082, </IMG>, EOS]
          └── text tokens ──┘          └────── image tokens ───────┘
```

In [ ]:
class UnifiedVocabulary:
    """
    Manages a unified vocabulary spanning text and image tokens.

    The vocabulary is partitioned into contiguous ranges:
      [0, text_vocab_size)                                → text tokens
      [text_vocab_size, text_vocab_size + image_codebook)  → image tokens
      [text_vocab_size + image_codebook, ...]              → special tokens

    This flat layout means a single embedding table and softmax can
    handle all modalities — no modality-specific heads needed.
    """

    def __init__(
        self,
        text_vocab_size: int = 32000,
        image_codebook_size: int = 8192,
        audio_codebook_size: int = 4096,
    ):
        self.text_vocab_size = text_vocab_size
        self.image_codebook_size = image_codebook_size
        self.audio_codebook_size = audio_codebook_size

        # Compute offsets for each modality's range in the unified vocab
        self.image_offset = text_vocab_size
        self.audio_offset = text_vocab_size + image_codebook_size

        # Special tokens placed after all modality tokens
        special_start = self.audio_offset + audio_codebook_size
        self.special_tokens = {
            "<BOS>": special_start,
            "<EOS>": special_start + 1,
            "<IMG>": special_start + 2,
            "</IMG>": special_start + 3,
            "<AUD>": special_start + 4,
            "</AUD>": special_start + 5,
            "<PAD>": special_start + 6,
        }

        self.total_vocab_size = special_start + len(self.special_tokens)

    def text_to_unified(self, text_ids: torch.Tensor) -> torch.Tensor:
        """Text token IDs are already in [0, text_vocab_size), no offset needed."""
        return text_ids

    def image_to_unified(self, image_codes: torch.Tensor) -> torch.Tensor:
        """Shift image codebook indices into the unified vocabulary range."""
        return image_codes + self.image_offset

    def audio_to_unified(self, audio_codes: torch.Tensor) -> torch.Tensor:
        """Shift audio codebook indices into the unified vocabulary range."""
        return audio_codes + self.audio_offset

    def unified_to_modality(
        self, token_ids: torch.Tensor
    ) -> Tuple[str, torch.Tensor]:
        """
        Determine which modality a token belongs to and return the
        modality-local index.
        """
        # Works element-wise for analysis; for batched ops, use masks
        if token_ids.max() < self.text_vocab_size:
            return "text", token_ids
        elif token_ids.max() < self.image_offset + self.image_codebook_size:
            return "image", token_ids - self.image_offset
        else:
            return "audio", token_ids - self.audio_offset

    def create_interleaved_sequence(
        self,
        text_ids: torch.Tensor,
        image_codes: Optional[torch.Tensor] = None,
        audio_codes: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Build a unified token sequence: <BOS> text <IMG> image </IMG> [<AUD> audio </AUD>] <EOS>

        This is the format used by Chameleon for interleaved text-image data.
        """
        parts = [torch.tensor([self.special_tokens["<BOS>"]])]

        # Add text tokens
        parts.append(self.text_to_unified(text_ids))

        # Add image tokens with delimiters
        if image_codes is not None:
            parts.append(torch.tensor([self.special_tokens["<IMG>"]]))
            parts.append(self.image_to_unified(image_codes.flatten()))
            parts.append(torch.tensor([self.special_tokens["</IMG>"]]))

        # Add audio tokens with delimiters
        if audio_codes is not None:
            parts.append(torch.tensor([self.special_tokens["<AUD>"]]))
            parts.append(self.audio_to_unified(audio_codes.flatten()))
            parts.append(torch.tensor([self.special_tokens["</AUD>"]]))

        parts.append(torch.tensor([self.special_tokens["<EOS>"]]))

        return torch.cat(parts)

    def get_modality_mask(self, token_ids: torch.Tensor) -> Dict[str, torch.Tensor]:
        """
        Return boolean masks indicating which positions are text, image, audio.
        Useful for applying modality-specific losses or attention patterns.
        """
        return {
            "text": token_ids < self.text_vocab_size,
            "image": (token_ids >= self.image_offset)
            & (token_ids < self.image_offset + self.image_codebook_size),
            "audio": (token_ids >= self.audio_offset)
            & (token_ids < self.audio_offset + self.audio_codebook_size),
            "special": token_ids >= self.audio_offset + self.audio_codebook_size,
        }


# Demonstrate the unified vocabulary
vocab = UnifiedVocabulary(
    text_vocab_size=32000, image_codebook_size=8192, audio_codebook_size=4096
)

print(f"Total vocabulary size: {vocab.total_vocab_size:,}")
print(f"\nVocabulary layout:")
print(f"  Text:    [0, {vocab.text_vocab_size})")
print(f"  Image:   [{vocab.image_offset}, {vocab.image_offset + vocab.image_codebook_size})")
print(f"  Audio:   [{vocab.audio_offset}, {vocab.audio_offset + vocab.audio_codebook_size})")
print(f"  Special: {vocab.special_tokens}")

# Create a sample interleaved sequence
fake_text = torch.tensor([65, 2910, 315, 64, 4529])
fake_image = torch.tensor([4021, 7193, 301, 5544, 1082])

unified_seq = vocab.create_interleaved_sequence(fake_text, fake_image)
print(f"\nInterleaved sequence: {unified_seq.tolist()}")
print(f"Sequence length: {len(unified_seq)}")

# Show modality masks
masks = vocab.get_modality_mask(unified_seq)
for mod, mask in masks.items():
    positions = torch.where(mask)[0].tolist()
    if positions:
        print(f"  {mod:8s} tokens at positions: {positions}")

# 5) Interleaved Training Data

## 5.1 Why Interleaving Matters

The power of native multimodal models comes from training on **naturally interleaved** data — web pages where text and images co-occur in meaningful relationships.

```
Example web document:
┌─────────────────────────────────────────────────┐
│  "The Eiffel Tower was built in 1889."          │  ← text tokens
│  [IMAGE: photo of Eiffel Tower]                 │  ← image tokens
│  "It stands 330 meters tall and is located"     │  ← text tokens
│  "in the Champ de Mars in Paris."               │  ← text tokens
│  [IMAGE: aerial view of Champ de Mars]          │  ← image tokens
│  "The tower was designed by Gustave Eiffel."    │  ← text tokens
└─────────────────────────────────────────────────┘

Tokenized sequence:
<BOS> The Eif fel Tower was built in 1889 .
      <IMG> [256 image tokens] </IMG>
      It stands 330 meters tall and is located
      in the Champ de Mars in Paris .
      <IMG> [256 image tokens] </IMG>
      The tower was designed by Gust ave Eif fel .
<EOS>
```

### Chameleon's Training Data Mix
- **Text-only:** 2.9T tokens from web crawl, books, code
- **Text-image interleaved:** 1.5T tokens from web documents with images
- **Image-text pairs:** 400M paired samples (similar to LAION)

### Why Not Just Pairs?
Paired data (one caption per image) teaches **alignment** but not **reasoning across modalities.** Interleaved data teaches the model that:
- Images can appear mid-sentence
- Multiple images can relate to the same narrative
- Text can reference specific visual details

## 5.2 Simulating Interleaved Data

Since we don't have a web crawl, we'll simulate the data format using CIFAR-10 with synthetic captions.

In [ ]:
class InterleavedDataGenerator:
    """
    Generates synthetic interleaved text-image sequences for demonstration.

    In production (Chameleon, Emu3), this data comes from:
    1. Common Crawl web pages with inline images
    2. Wikipedia articles
    3. Instruction-following datasets
    4. Image-caption pairs (LAION, CC12M)

    The generator creates sequences in the format:
      <BOS> [text] <IMG> [image tokens] </IMG> [text] ... <EOS>
    """

    # Simple synthetic captions for CIFAR-10 classes
    TEMPLATES = {
        0: "A photograph of an airplane flying through the sky.",
        1: "A photo of a car parked on the street.",
        2: "A picture of a bird perched on a branch.",
        3: "An image of a cat sitting on a cushion.",
        4: "A photograph of a deer in a forest.",
        5: "A photo of a dog running in the park.",
        6: "An image of a frog on a lily pad.",
        7: "A picture of a horse galloping in a field.",
        8: "A photograph of a ship sailing on the ocean.",
        9: "A photo of a truck driving on the highway.",
    }

    def __init__(
        self,
        vocab: UnifiedVocabulary,
        vqvae_model: VQVAE,
        max_text_tokens: int = 20,
        device: torch.device = DEVICE,
    ):
        self.vocab = vocab
        self.vqvae = vqvae_model
        self.max_text_tokens = max_text_tokens
        self.device = device

    def _fake_text_tokens(self, label: int) -> torch.Tensor:
        """
        Generate pseudo text token IDs from a template.
        In practice, these would come from a real BPE tokenizer.
        We simulate by hashing characters to token IDs.
        """
        text = self.TEMPLATES[label]
        # Simple character-level hashing to simulate BPE tokens
        tokens = []
        words = text.split()
        for word in words[: self.max_text_tokens]:
            token_id = hash(word) % self.vocab.text_vocab_size
            tokens.append(abs(token_id))
        return torch.tensor(tokens, dtype=torch.long)

    @torch.no_grad()
    def generate_sequence(
        self, image: torch.Tensor, label: int
    ) -> Tuple[torch.Tensor, Dict]:
        """
        Create a single interleaved sequence from an image and its label.

        Returns:
            unified_tokens: (seq_len,) tensor of unified token IDs
            metadata: dict with modality masks and original image
        """
        # Encode image to discrete tokens
        # (1, 3, H, W) → (1, h, w)
        image_input = image.unsqueeze(0).to(self.device)
        codes = self.vqvae.encode(image_input)  # (1, h, w)
        image_tokens = codes.squeeze(0).cpu()  # (h, w)

        # Generate text tokens
        text_tokens = self._fake_text_tokens(label)

        # Build unified sequence
        unified = self.vocab.create_interleaved_sequence(text_tokens, image_tokens)

        metadata = {
            "masks": self.vocab.get_modality_mask(unified),
            "image_shape": image_tokens.shape,
            "text_length": len(text_tokens),
        }

        return unified, metadata


# Create generator and produce a sample sequence
generator = InterleavedDataGenerator(vocab, vqvae)

sample_image, sample_label = train_dataset[0]
seq, meta = generator.generate_sequence(sample_image, sample_label)

print(f"Interleaved sequence length: {len(seq)} tokens")
print(f"  Text tokens:    {meta['masks']['text'].sum().item()}")
print(f"  Image tokens:   {meta['masks']['image'].sum().item()}")
print(f"  Special tokens: {meta['masks']['special'].sum().item()}")
print(f"\nFirst 10 tokens:    {seq[:10].tolist()}")
print(f"Token range:        [{seq.min().item()}, {seq.max().item()}]")

# 6) Architecture Options: The Three Paradigms

## 6.1 Overview

The field has converged on three main architectural approaches for native multimodal models. Each makes different tradeoffs between simplicity, quality, and flexibility.

```
┌──────────────────────────────────────────────────────────────────────┐
│                    APPROACH COMPARISON                                │
├──────────────────┬───────────────┬──────────────┬────────────────────┤
│                  │ Pure AR       │ Transfusion  │ Decoupled (Janus)  │
│                  │ (Chameleon,   │ (Meta 2024)  │ (DeepSeek 2025)    │
│                  │  Emu3)        │              │                    │
├──────────────────┼───────────────┼──────────────┼────────────────────┤
│ Text loss        │ Cross-entropy │ Cross-entropy│ Cross-entropy      │
│ Image loss       │ Cross-entropy │ Diffusion    │ Cross-entropy      │
│ Image repr.      │ VQ tokens     │ Continuous   │ VQ tokens (gen)    │
│                  │               │ patches      │ SigLIP (understand)│
│ Generation       │ Token-by-token│ Parallel     │ Token-by-token     │
│                  │ (sequential)  │ (diffusion)  │ (sequential)       │
│ Simplicity       │ ★★★          │ ★★           │ ★★                 │
│ Image quality    │ ★★           │ ★★★          │ ★★★                │
│ Understanding    │ ★★           │ ★★           │ ★★★                │
│ Unified training │ ★★★          │ ★★           │ ★                  │
└──────────────────┴───────────────┴──────────────┴────────────────────┘
```

## 6.2 Approach A: Pure Autoregressive (Chameleon, Emu3)

**Core idea:** Treat ALL modalities as discrete token sequences. One model, one loss (cross-entropy), one generation procedure (autoregressive sampling).

```
                    Unified token sequence
Input:   <BOS> The cat sat <IMG> [t1 t2 ... t256] </IMG> on the mat <EOS>
                 ↓                    ↓                      ↓
         Text embeddings    Image embeddings         Text embeddings
                 ↓                    ↓                      ↓
         ┌────────────────────────────────────────────────────┐
         │              Transformer (causal attention)         │
         │   Every token attends to all previous tokens        │
         │   regardless of modality                            │
         └────────────────────────────────────────────────────┘
                 ↓                    ↓                      ↓
         Predict next token (cross-entropy over unified vocab)
```

**Chameleon-specific innovations:**
1. **QK-Norm:** Normalize query and key vectors before attention to prevent entropy collapse when mixing modalities with very different embedding scales
2. **Dropout on image tokens:** Prevents the model from "cheating" by copying adjacent image tokens
3. **Balanced loss weighting:** Image tokens are ~4× more frequent; without careful weighting, text quality degrades

In [ ]:
class QKNorm(nn.Module):
    """
    Query-Key Normalization from Chameleon (Section 3.2).

    Problem: When text and image tokens share the same attention mechanism,
    their embedding magnitudes can differ by 10-100×. This causes attention
    logits to saturate for one modality while being too flat for another.

    Solution: L2-normalize Q and K before computing attention, then scale
    by a learned temperature. This ensures attention dynamics are consistent
    across modalities regardless of embedding scale.

    Without QK-Norm, Chameleon found training diverges around 1B tokens
    when mixing text and image data.
    """

    def __init__(self, head_dim: int):
        super().__init__()
        # Learned temperature per head — initialized to sqrt(head_dim)
        # to match standard scaled dot-product attention at initialization
        self.scale = nn.Parameter(torch.ones(1) * math.sqrt(head_dim))

    def forward(
        self, q: torch.Tensor, k: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            q: (batch_num, num_heads, seq_len, head_dim)
            k: (batch_num, num_heads, seq_len, head_dim)
        Returns:
            Normalized q, k with same shapes
        """
        # L2-normalize along the head dimension
        q = F.normalize(q, dim=-1)
        k = F.normalize(k, dim=-1)

        # Scale by learned temperature
        # After normalization, q·k ∈ [-1, 1]; scale controls sharpness
        q = q * self.scale
        return q, k


class MultimodalCausalAttention(nn.Module):
    """
    Causal multi-head attention with optional QK-Norm.

    This is the core attention mechanism used in autoregressive multimodal
    models. Key property: text tokens can attend to preceding image tokens
    and vice versa, enabling cross-modal reasoning.
    """

    def __init__(
        self,
        model_dim: int = 512,
        num_heads: int = 8,
        use_qk_norm: bool = True,
    ):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = model_dim // num_heads

        self.qkv_proj = nn.Linear(model_dim, 3 * model_dim, bias=False)
        self.out_proj = nn.Linear(model_dim, model_dim, bias=False)
        self.qk_norm = QKNorm(self.head_dim) if use_qk_norm else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (batch_num, seq_len, model_dim)
        Returns:
            (batch_num, seq_len, model_dim)
        """
        batch_num, seq_len, model_dim = x.shape

        # Project to Q, K, V
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, 3 * model_dim)
        qkv = self.qkv_proj(x)

        # Split into Q, K, V and reshape for multi-head attention
        # (batch_num, seq_len, 3*model_dim) → 3 × (batch_num, num_heads, seq_len, head_dim)
        qkv = rearrange(
            qkv, "b s (three h d) -> three b h s d", three=3, h=self.num_heads
        )
        q, k, v = qkv[0], qkv[1], qkv[2]

        # Apply QK-Norm if enabled (critical for multimodal stability)
        if self.qk_norm is not None:
            q, k = self.qk_norm(q, k)

        # Scaled dot-product attention with causal mask
        # (batch_num, num_heads, seq_len, head_dim) × (batch_num, num_heads, head_dim, seq_len)
        # → (batch_num, num_heads, seq_len, seq_len)
        scale = 1.0 if self.qk_norm else (1.0 / math.sqrt(self.head_dim))
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale

        # Causal mask: prevent attending to future tokens
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool), diagonal=1
        )
        attn = attn.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float("-inf"))

        attn = F.softmax(attn, dim=-1)

        # (batch_num, num_heads, seq_len, seq_len) × (batch_num, num_heads, seq_len, head_dim)
        # → (batch_num, num_heads, seq_len, head_dim)
        out = torch.matmul(attn, v)

        # Merge heads
        # (batch_num, num_heads, seq_len, head_dim) → (batch_num, seq_len, model_dim)
        out = rearrange(out, "b h s d -> b s (h d)")

        # Output projection
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, model_dim)
        return self.out_proj(out)


# Verify attention
attn = MultimodalCausalAttention(model_dim=256, num_heads=4, use_qk_norm=True)
x = torch.randn(2, 32, 256)
out = attn(x)
print(f"Attention input: {x.shape} → output: {out.shape}")

## 6.3 Approach B: Transfusion (Meta 2024)

**Key insight:** Text is inherently sequential (word order matters), but images are inherently 2D and spatially coherent. Using cross-entropy loss for image tokens forces a left-to-right generation order that doesn't match how images are structured.

**Solution:** Use **autoregressive cross-entropy** for text and **diffusion loss** for images — within the same model and the same forward pass.

```
Input sequence:
  <BOS> The cat sat <IMG> [continuous image patches] </IMG> on the mat <EOS>

Loss computation:
  Text tokens  → Standard next-token cross-entropy
  Image tokens → Diffusion loss (predict noise added to image patches)

Generation:
  Text  → Sample autoregressively (token by token)
  Image → At <IMG> position, run denoising diffusion (parallel across patches)
```

**Why this helps:**
1. No VQ-VAE needed for images → no information bottleneck from discretization
2. Image generation is **parallel** across spatial positions (faster than AR)
3. Diffusion naturally handles the continuous, spatially-coherent nature of images

**Tradeoff:** More complex training (two loss functions) and generation (two procedures).

In [ ]:
class TransfusionBlock(nn.Module):
    """
    Simplified Transfusion transformer block.

    Key difference from standard transformer: dual loss computation.
    The same hidden states are used for:
      - Cross-entropy prediction of text tokens
      - Noise prediction (diffusion) for image patches

    This is implemented by having the attention operate over ALL tokens
    but applying different loss functions based on modality.

    Architecture per block:
      x → LayerNorm → CausalAttention → + residual
        → LayerNorm → FFN → + residual
    """

    def __init__(self, model_dim: int = 512, num_heads: int = 8):
        super().__init__()
        self.norm1 = nn.LayerNorm(model_dim)
        self.attn = MultimodalCausalAttention(model_dim, num_heads, use_qk_norm=True)
        self.norm2 = nn.LayerNorm(model_dim)
        self.ffn = nn.Sequential(
            nn.Linear(model_dim, model_dim * 4),
            nn.GELU(),
            nn.Linear(model_dim * 4, model_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-norm residual connections
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, model_dim)
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x


class TransfusionModel(nn.Module):
    """
    Simplified Transfusion architecture demonstrating dual-loss training.

    For text: standard autoregressive LM with cross-entropy loss
    For images: predicts the noise ε added to diffusion-noised image patches

    The key implementation detail is the LOSS MASKING:
    - Text positions contribute to cross-entropy loss
    - Image positions contribute to diffusion loss
    - The transformer backbone is shared; only the loss heads differ
    """

    def __init__(
        self,
        vocab_size: int = 32000,
        model_dim: int = 512,
        num_heads: int = 8,
        num_layers: int = 4,
        image_patch_dim: int = 768,
    ):
        super().__init__()
        self.model_dim = model_dim

        # Shared token embedding for text
        self.text_embed = nn.Embedding(vocab_size, model_dim)

        # Projection for continuous image patches (no discrete tokenization)
        # (batch_num, num_patches, image_patch_dim) → (batch_num, num_patches, model_dim)
        self.image_proj_in = nn.Linear(image_patch_dim, model_dim)

        # Diffusion timestep embedding (sinusoidal)
        self.time_embed = nn.Sequential(
            nn.Linear(model_dim, model_dim),
            nn.SiLU(),
            nn.Linear(model_dim, model_dim),
        )

        # Shared transformer backbone
        self.blocks = nn.ModuleList(
            [TransfusionBlock(model_dim, num_heads) for _ in range(num_layers)]
        )
        self.final_norm = nn.LayerNorm(model_dim)

        # Text prediction head (cross-entropy)
        self.text_head = nn.Linear(model_dim, vocab_size)

        # Image noise prediction head (diffusion)
        # Predicts the noise ε that was added to the image patches
        self.image_head = nn.Linear(model_dim, image_patch_dim)

    def _sinusoidal_embed(self, t: torch.Tensor) -> torch.Tensor:
        """
        Sinusoidal positional encoding for diffusion timesteps.
        (batch_num,) → (batch_num, model_dim)
        """
        half_dim = self.model_dim // 2
        freq = torch.exp(
            -math.log(10000.0)
            * torch.arange(half_dim, device=t.device).float()
            / half_dim
        )
        # (batch_num, 1) × (half_dim,) → (batch_num, half_dim)
        args = t.unsqueeze(1) * freq.unsqueeze(0)
        # (batch_num, model_dim)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

    def forward(
        self,
        text_ids: torch.Tensor,
        image_patches: Optional[torch.Tensor] = None,
        is_image_mask: Optional[torch.Tensor] = None,
        diffusion_timestep: Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass computing both text and image losses.

        Args:
            text_ids:          (batch_num, text_len) — text token IDs
            image_patches:     (batch_num, num_patches, image_patch_dim) — noised image patches
            is_image_mask:     (batch_num, seq_len) — True for image positions
            diffusion_timestep: (batch_num,) — noise level for diffusion

        Returns:
            Dict with 'text_logits' and 'noise_pred'
        """
        # Embed text tokens
        # (batch_num, text_len) → (batch_num, text_len, model_dim)
        text_embeds = self.text_embed(text_ids)

        if image_patches is not None:
            # Project image patches to model dim
            # (batch_num, num_patches, image_patch_dim) → (batch_num, num_patches, model_dim)
            image_embeds = self.image_proj_in(image_patches)

            # Add diffusion timestep information to image embeddings
            if diffusion_timestep is not None:
                t_embed = self.time_embed(self._sinusoidal_embed(diffusion_timestep.float()))
                # (batch_num, model_dim) → (batch_num, 1, model_dim) broadcast to image positions
                image_embeds = image_embeds + t_embed.unsqueeze(1)

            # Interleave: for simplicity, concatenate [text, image]
            # (batch_num, text_len + num_patches, model_dim)
            hidden = torch.cat([text_embeds, image_embeds], dim=1)
        else:
            hidden = text_embeds

        # Pass through shared transformer backbone
        for block in self.blocks:
            # (batch_num, seq_len, model_dim) → (batch_num, seq_len, model_dim)
            hidden = block(hidden)
        hidden = self.final_norm(hidden)

        results = {}

        # Text prediction: cross-entropy logits
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, vocab_size)
        results["text_logits"] = self.text_head(hidden[:, : text_ids.shape[1]])

        # Image prediction: noise estimate for diffusion
        if image_patches is not None:
            # (batch_num, num_patches, model_dim) → (batch_num, num_patches, image_patch_dim)
            results["noise_pred"] = self.image_head(hidden[:, text_ids.shape[1] :])

        return results


# Demonstrate Transfusion
transfusion = TransfusionModel(vocab_size=32000, model_dim=256, num_heads=4, num_layers=2)

# Simulate inputs
batch_num = 2
text_ids = torch.randint(0, 32000, (batch_num, 20))
image_patches = torch.randn(batch_num, 64, 768)
timesteps = torch.randint(0, 1000, (batch_num,))

out = transfusion(text_ids, image_patches, diffusion_timestep=timesteps)
print(f"Text logits shape:  {out['text_logits'].shape}")
print(f"Noise pred shape:   {out['noise_pred'].shape}")

print(f"\nTransfusion computes two losses in one forward pass:")
print(f"  Text:  cross_entropy(logits, target_text_ids)")
print(f"  Image: mse(noise_pred, actual_noise_added)")

## 6.4 Approach C: Janus Pro — Decoupled Visual Encoding (DeepSeek 2025)

**The fundamental insight:** Understanding images and generating images require **different visual representations.**

- **Understanding** needs high-level **semantic** features: "this is a dog," "the scene is outdoors"
  → Best served by contrastive vision encoders (SigLIP, CLIP) trained on billions of image-text pairs
  
- **Generation** needs low-level **pixel** details: "the fur texture is soft and brown," "the background has a gradient from blue to orange"
  → Best served by VQ tokenizers that preserve spatial/textural information

**Previous approaches (Chameleon, Emu3) used the same VQ tokenizer for both**, which creates a conflict:
- Making VQ tokens more semantic → better understanding, worse generation
- Making VQ tokens more pixel-detailed → better generation, worse understanding

**Janus Pro's solution: separate encoding paths, shared LLM backbone.**

```
Understanding path:                     Generation path:
  Image                                   Text prompt
    ↓                                       ↓
  SigLIP encoder                          LLM generates image tokens
    ↓                                       ↓
  Adaptor MLP                             VQ codebook lookup
    ↓                                       ↓
  LLM processes (with text)               VQ Decoder → Image
    ↓
  Text output

Key: The LLM is SHARED — it sees different representations depending on the task.
```

### Why This Works

SigLIP embeddings are trained with contrastive loss on 4B+ image-text pairs → excellent at capturing "what's in this image." VQ-VAE tokens are trained with reconstruction loss → excellent at capturing "what does this image look like pixel-by-pixel."

By routing through different encoders, each task gets the representation it needs, while the shared LLM provides reasoning and language capabilities to both paths.

In [ ]:
class JanusProArchitecture(nn.Module):
    """
    Simplified Janus Pro architecture demonstrating decoupled visual encoding.

    Understanding path: SigLIP-style encoder → adaptor → LLM
    Generation path:    LLM → VQ tokens → VQ decoder

    The key design: same LLM backbone, different visual encoders for
    understanding vs generation. This avoids the representation conflict
    where a single tokenizer must serve both semantic understanding
    and pixel-level generation.
    """

    def __init__(
        self,
        model_dim: int = 512,
        num_heads: int = 8,
        num_layers: int = 4,
        text_vocab_size: int = 32000,
        image_codebook_size: int = 8192,
        siglip_dim: int = 768,
    ):
        super().__init__()
        self.model_dim = model_dim

        # === UNDERSTANDING PATH ===
        # SigLIP-style vision encoder (frozen in practice — we simulate with a MLP)
        # In production: SigLIP-400M or SigLIP-SO400M
        self.understanding_encoder = nn.Sequential(
            nn.Linear(3 * 14 * 14, siglip_dim),  # Simplified: flatten patch
            nn.GELU(),
            nn.Linear(siglip_dim, siglip_dim),
        )
        # Adaptor: project SigLIP features to LLM input space
        # (batch_num, num_patches, siglip_dim) → (batch_num, num_patches, model_dim)
        self.understanding_adaptor = nn.Sequential(
            nn.Linear(siglip_dim, model_dim),
            nn.GELU(),
            nn.Linear(model_dim, model_dim),
        )

        # === GENERATION PATH ===
        # VQ codebook embedding for generation (separate from understanding)
        self.generation_embed = nn.Embedding(image_codebook_size, model_dim)
        # Generation head: predict VQ codebook indices
        self.generation_head = nn.Linear(model_dim, image_codebook_size)

        # === SHARED LLM BACKBONE ===
        self.text_embed = nn.Embedding(text_vocab_size, model_dim)
        self.blocks = nn.ModuleList(
            [TransfusionBlock(model_dim, num_heads) for _ in range(num_layers)]
        )
        self.final_norm = nn.LayerNorm(model_dim)
        self.text_head = nn.Linear(model_dim, text_vocab_size)

    def understand(
        self,
        text_ids: torch.Tensor,
        image_features: torch.Tensor,
    ) -> torch.Tensor:
        """
        Understanding path: process image with SigLIP encoder,
        then reason with LLM.

        Args:
            text_ids:       (batch_num, text_len) — question/instruction tokens
            image_features: (batch_num, num_patches, siglip_dim) — SigLIP features

        Returns:
            text_logits: (batch_num, text_len, text_vocab_size)
        """
        # Project SigLIP features through adaptor
        # (batch_num, num_patches, siglip_dim) → (batch_num, num_patches, model_dim)
        visual_embeds = self.understanding_adaptor(image_features)

        # Embed text
        # (batch_num, text_len) → (batch_num, text_len, model_dim)
        text_embeds = self.text_embed(text_ids)

        # Concatenate: [visual_context, text_query]
        # (batch_num, num_patches + text_len, model_dim)
        hidden = torch.cat([visual_embeds, text_embeds], dim=1)

        # Process through shared LLM
        for block in self.blocks:
            hidden = block(hidden)
        hidden = self.final_norm(hidden)

        # Only predict text tokens (understanding → text output)
        text_hidden = hidden[:, image_features.shape[1] :]
        return self.text_head(text_hidden)

    def generate_image_tokens(
        self, text_ids: torch.Tensor
    ) -> torch.Tensor:
        """
        Generation path: from text prompt, autoregressively predict
        VQ codebook indices.

        Args:
            text_ids: (batch_num, text_len) — generation prompt tokens

        Returns:
            logits: (batch_num, text_len, image_codebook_size) — next image token probs
        """
        # Embed text prompt
        # (batch_num, text_len) → (batch_num, text_len, model_dim)
        hidden = self.text_embed(text_ids)

        # Process through shared LLM
        for block in self.blocks:
            hidden = block(hidden)
        hidden = self.final_norm(hidden)

        # Predict image tokens (generation → VQ codes)
        # (batch_num, text_len, model_dim) → (batch_num, text_len, image_codebook_size)
        return self.generation_head(hidden)


# Demonstrate both paths
janus = JanusProArchitecture(model_dim=256, num_heads=4, num_layers=2)

# Understanding path
text_ids = torch.randint(0, 32000, (2, 10))
siglip_features = torch.randn(2, 16, 768)
understand_logits = janus.understand(text_ids, siglip_features)
print(f"Understanding path:")
print(f"  Input: text {text_ids.shape} + SigLIP features {siglip_features.shape}")
print(f"  Output: text logits {understand_logits.shape}")

# Generation path
gen_logits = janus.generate_image_tokens(text_ids)
print(f"\nGeneration path:")
print(f"  Input: text prompt {text_ids.shape}")
print(f"  Output: image token logits {gen_logits.shape}")

print(f"\nKey insight: SAME LLM backbone ({sum(p.numel() for p in janus.blocks.parameters()):,} params)")
print(f"but DIFFERENT visual representations for each task.")

## 6.5 Deep Comparison: When to Use Which Approach

### The Representation Conflict (Quantified)

Janus Pro's paper provides direct evidence for the representation conflict. When using the same VQ tokenizer for both tasks:

| Metric | Shared VQ | Janus (Decoupled) | Δ |
|--------|-----------|-------------------|---|
| GenEval (generation quality) | 0.72 | 0.80 | +11% |
| MMBench (understanding) | 69.4 | 79.2 | +14% |

The improvement in **both** tasks when using separate encoders proves that the conflict is real — a single tokenizer cannot optimally serve both objectives.

### Computational Comparison

```
Chameleon 34B:
  - Text: 2.9T tokens @ 4096 seq_len
  - Image: 1024 tokens per image (VQ with 32×32 grid)
  - Total training: ~10M GPU hours on A100s
  - Generation: 1024 AR steps per image (slow)

Transfusion 7B:
  - Text: AR as usual
  - Image: 256 latent patches, ~50 diffusion steps
  - Generation: 50 denoising steps (parallel across patches, much faster)
  - But: image tokens don't compress into a neat sequence for the LLM

Janus Pro 7B:
  - Understanding: 576 SigLIP tokens per image
  - Generation: 576 VQ tokens per image
  - Competitive with both specialized understanding AND generation models
  - Key advantage: can use different resolutions for each path
```

### Practical Recommendations

- **If you need maximum simplicity and a truly unified model:** Chameleon/Emu3 approach
- **If you need the best image generation quality:** Transfusion (diffusion for images)
- **If you need excellent understanding AND generation:** Janus Pro (decoupled encoders)
- **If you're building a product:** Janus Pro is currently the best quality-per-parameter

# 7) Simplified Any-to-Any Demo: Text → Image Tokens → Image

## 7.1 The Complete Pipeline

Now we'll connect all the pieces: use a tiny autoregressive model to generate image tokens conditioned on text, then decode those tokens back to an image using our VQ-VAE decoder.

```
Pipeline:
  "a red circle"
       ↓
  Text tokenizer (simulated)
       ↓
  [text_token_1, text_token_2, ..., <IMG>]
       ↓
  Tiny autoregressive LM: predict image tokens one by one
       ↓
  [img_token_1, img_token_2, ..., img_token_256]
       ↓
  VQ-VAE decoder: codebook lookup → decode to pixels
       ↓
  Generated image (128×128×3)
```

This is a **toy** demonstration — the generated images won't look realistic because our model is tiny and trained on minimal data. But it shows the complete mechanism that Chameleon and Emu3 use at scale.

### Sample Input/Output

```
Input:  text prompt "a photo of a cat" → text tokens [65, 2910, ...]
Output: 256 generated image tokens → VQ-VAE decode → (1, 3, 128, 128) image
```

In [ ]:
class TinyMultimodalLM(nn.Module):
    """
    Minimal autoregressive language model for text-to-image generation.

    This demonstrates the Chameleon/Emu3 generation paradigm:
    1. Encode text prompt as tokens
    2. Autoregressively generate image tokens (one at a time)
    3. Decode image tokens via VQ-VAE decoder

    Deliberately small for demonstration; real models are 7B-34B parameters.
    """

    def __init__(
        self,
        total_vocab_size: int,
        model_dim: int = 256,
        num_heads: int = 4,
        num_layers: int = 4,
        max_seq_len: int = 512,
    ):
        super().__init__()
        self.model_dim = model_dim
        self.max_seq_len = max_seq_len

        self.token_embed = nn.Embedding(total_vocab_size, model_dim)

        # Learned positional embeddings
        self.pos_embed = nn.Embedding(max_seq_len, model_dim)

        # Transformer blocks
        self.blocks = nn.ModuleList(
            [TransfusionBlock(model_dim, num_heads) for _ in range(num_layers)]
        )
        self.final_norm = nn.LayerNorm(model_dim)
        self.lm_head = nn.Linear(model_dim, total_vocab_size)

    def forward(
        self, token_ids: torch.Tensor
    ) -> torch.Tensor:
        """
        Standard autoregressive forward pass.

        Args:
            token_ids: (batch_num, seq_len) — unified token IDs

        Returns:
            logits: (batch_num, seq_len, total_vocab_size)
        """
        batch_num, seq_len = token_ids.shape

        # Token + positional embeddings
        # (batch_num, seq_len) → (batch_num, seq_len, model_dim)
        positions = torch.arange(seq_len, device=token_ids.device)
        hidden = self.token_embed(token_ids) + self.pos_embed(positions)

        # Transformer blocks
        for block in self.blocks:
            hidden = block(hidden)
        hidden = self.final_norm(hidden)

        # Project to vocabulary logits
        # (batch_num, seq_len, model_dim) → (batch_num, seq_len, total_vocab_size)
        return self.lm_head(hidden)

    @torch.no_grad()
    def generate_image_tokens(
        self,
        text_prefix: torch.Tensor,
        img_start_token: int,
        img_end_token: int,
        image_vocab_start: int,
        image_vocab_end: int,
        num_image_tokens: int = 256,
        temperature: float = 1.0,
        top_k: int = 50,
    ) -> torch.Tensor:
        """
        Autoregressively generate image tokens given a text prefix.

        Args:
            text_prefix:      (1, prefix_len) — text tokens + <IMG> token
            img_start_token:  ID of <IMG> token (already in prefix)
            img_end_token:    ID of </IMG> token
            image_vocab_start: First token ID in image vocabulary range
            image_vocab_end:   Last token ID in image vocabulary range
            num_image_tokens:  How many image tokens to generate
            temperature:       Sampling temperature
            top_k:             Top-k filtering

        Returns:
            generated: (1, num_image_tokens) — generated image token IDs (in unified vocab)
        """
        device = text_prefix.device
        current_seq = text_prefix.clone()

        generated_tokens = []

        for _ in range(num_image_tokens):
            # Truncate to max sequence length
            if current_seq.shape[1] > self.max_seq_len:
                current_seq = current_seq[:, -self.max_seq_len :]

            # Forward pass
            # (1, current_len) → (1, current_len, total_vocab_size)
            logits = self.forward(current_seq)

            # Take logits for the last position
            # (1, total_vocab_size)
            next_logits = logits[:, -1, :] / temperature

            # Restrict to image vocabulary only
            # Set all non-image token logits to -inf
            mask = torch.ones_like(next_logits) * float("-inf")
            mask[:, image_vocab_start:image_vocab_end] = 0.0
            next_logits = next_logits + mask

            # Top-k filtering
            if top_k > 0:
                top_k_vals, _ = torch.topk(next_logits, top_k, dim=-1)
                threshold = top_k_vals[:, -1:]
                next_logits[next_logits < threshold] = float("-inf")

            # Sample
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            generated_tokens.append(next_token)

            # Append to sequence
            current_seq = torch.cat([current_seq, next_token], dim=1)

        # (1, num_image_tokens)
        return torch.cat(generated_tokens, dim=1)


# Instantiate the tiny LM
lm = TinyMultimodalLM(
    total_vocab_size=vocab.total_vocab_size,
    model_dim=256,
    num_heads=4,
    num_layers=4,
    max_seq_len=512,
).to(DEVICE)

print(f"Tiny LM parameters: {sum(p.numel() for p in lm.parameters()):,}")

In [ ]:
def train_tiny_lm(
    model: TinyMultimodalLM,
    vqvae_model: VQVAE,
    vocab: UnifiedVocabulary,
    dataloader: torch.utils.data.DataLoader,
    num_epochs: int = 2,
    lr: float = 3e-4,
    device: torch.device = DEVICE,
) -> List[float]:
    """
    Train the tiny LM on interleaved text-image sequences.

    The training objective is standard next-token prediction (cross-entropy)
    over the unified vocabulary — same loss for text and image tokens.
    This is the Chameleon/Emu3 training paradigm.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    generator = InterleavedDataGenerator(vocab, vqvae_model, device=device)
    losses = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        num_batches = 0

        pbar = tqdm(dataloader, desc=f"LM Epoch {epoch + 1}/{num_epochs}")
        for images, labels in pbar:
            images = images.to(device)

            # Generate interleaved sequences for the batch
            batch_seqs = []
            for i in range(images.shape[0]):
                seq, _ = generator.generate_sequence(images[i], labels[i].item())
                batch_seqs.append(seq)

            # Pad sequences to equal length
            max_len = min(max(len(s) for s in batch_seqs), model.max_seq_len)
            padded = torch.full(
                (len(batch_seqs), max_len),
                vocab.special_tokens["<PAD>"],
                dtype=torch.long,
            )
            for i, seq in enumerate(batch_seqs):
                length = min(len(seq), max_len)
                padded[i, :length] = seq[:length]

            padded = padded.to(device)

            # Standard language modeling: predict token at position t+1 from position t
            # Input: tokens [0, ..., T-1], Target: tokens [1, ..., T]
            input_ids = padded[:, :-1]
            target_ids = padded[:, 1:]

            logits = model(input_ids)

            # Cross-entropy loss (ignore padding tokens)
            loss = F.cross_entropy(
                logits.reshape(-1, logits.shape[-1]),
                target_ids.reshape(-1),
                ignore_index=vocab.special_tokens["<PAD>"],
            )

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            epoch_loss += loss.item()
            num_batches += 1
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_loss = epoch_loss / num_batches
        losses.append(avg_loss)
        print(f"Epoch {epoch + 1}: avg_loss={avg_loss:.4f}")

    return losses


# Train (limited epochs for demo speed)
lm_losses = train_tiny_lm(lm, vqvae, vocab, train_loader, num_epochs=2)

In [ ]:
def generate_and_visualize(
    lm_model: TinyMultimodalLM,
    vqvae_model: VQVAE,
    vocab: UnifiedVocabulary,
    text_label: int = 3,
    num_samples: int = 4,
    device: torch.device = DEVICE,
):
    """
    Complete text-to-image generation pipeline:
    1. Create text prompt tokens
    2. Autoregressively generate image tokens with the LM
    3. Decode image tokens to pixels with VQ-VAE

    Note: Results will be noisy/abstract since our model is tiny.
    The point is demonstrating the mechanism, not the quality.
    """
    lm_model.eval()
    vqvae_model.eval()

    generator = InterleavedDataGenerator(vocab, vqvae_model, device=device)
    text_tokens = generator._fake_text_tokens(text_label)

    fig, axes = plt.subplots(1, num_samples, figsize=(3 * num_samples, 3))

    for i in range(num_samples):
        # Build prefix: <BOS> text_tokens <IMG>
        prefix = torch.cat(
            [
                torch.tensor([vocab.special_tokens["<BOS>"]]),
                vocab.text_to_unified(text_tokens),
                torch.tensor([vocab.special_tokens["<IMG>"]]),
            ]
        ).unsqueeze(0).to(device)

        # Determine spatial dimensions of VQ-VAE output
        # For our VQ-VAE with 128×128 input and 3 downsamples: 16×16 grid
        spatial_h, spatial_w = 16, 16
        num_image_tokens = spatial_h * spatial_w

        # Generate image tokens autoregressively
        generated_unified = lm_model.generate_image_tokens(
            text_prefix=prefix,
            img_start_token=vocab.special_tokens["<IMG>"],
            img_end_token=vocab.special_tokens["</IMG>"],
            image_vocab_start=vocab.image_offset,
            image_vocab_end=vocab.image_offset + vocab.image_codebook_size,
            num_image_tokens=num_image_tokens,
            temperature=0.8,
            top_k=100,
        )

        # Convert from unified vocab IDs back to VQ-VAE codebook indices
        image_codes = generated_unified - vocab.image_offset
        image_codes = image_codes.clamp(0, vqvae_model.quantizer.codebook_size - 1)
        image_codes = image_codes.reshape(1, spatial_h, spatial_w)

        # Decode through VQ-VAE
        decoded = vqvae_model.decode(image_codes.to(device))

        # Visualize
        img = (decoded[0].cpu() * 0.5 + 0.5).clamp(0, 1).permute(1, 2, 0)
        axes[i].imshow(img.numpy())
        axes[i].set_title(f"Sample {i + 1}")
        axes[i].axis("off")

    class_name = InterleavedDataGenerator.TEMPLATES[text_label].split("of ")[-1].split(".")[0]
    plt.suptitle(
        f'Generated images for: "{class_name}"\n'
        f"(Tiny model — demonstrates mechanism, not quality)",
        fontsize=11,
    )
    plt.tight_layout()
    plt.show()

    print(f"\nPipeline summary:")
    print(f"  Text prompt → {len(text_tokens)} text tokens")
    print(f"  LM generated → {num_image_tokens} image tokens autoregressively")
    print(f"  VQ-VAE decoded → 128×128×3 RGB image")


generate_and_visualize(lm, vqvae, vocab, text_label=3, num_samples=4)

# 8) MiniCPM-o: A Real-World Any-to-Any Model in 8B Parameters

## 8.1 Architecture Overview

MiniCPM-o (OpenBMB, 2024-2025) demonstrates that any-to-any multimodal capability is achievable at a practical scale. It handles **4 modalities** — text, image, video, and audio — in a single 8B parameter model.

```
MiniCPM-o Architecture:
┌──────────────────────────────────────────────────────┐
│                    SigLIP-400M                        │  Vision encoder
│                    (image/video)                      │  (frozen, then fine-tuned)
└──────────┬───────────────────────────────────────────┘
           │ Visual features
           ▼
┌──────────────────────┐    ┌──────────────────────────┐
│   Perceiver Resampler│    │  Whisper-large-v3        │  Audio encoder
│   (compress tokens)  │    │  (audio → features)      │  (frozen)
└──────────┬───────────┘    └──────────┬───────────────┘
           │                           │
           ▼                           ▼
┌──────────────────────────────────────────────────────┐
│              Qwen2-7B LLM Backbone                    │  Shared LLM
│              (causal transformer)                      │
└──────────┬────────────────────────────┬──────────────┘
           │                            │
           ▼                            ▼
     Text output               Audio output (ChatTTS)
     (standard LM head)        (text → speech via TTS)
```

### Key Design Choices

1. **Encoder selection:** SigLIP for vision (understanding-focused), Whisper for audio (ASR-focused). This follows the Janus philosophy — use the best encoder for each modality.

2. **Token compression:** The Perceiver Resampler reduces 576+ visual tokens to a smaller set, keeping sequence lengths manageable. Critical for video (which otherwise would be thousands of tokens per second).

3. **End-to-end streaming:** MiniCPM-o supports real-time audio-visual conversation by processing audio chunks incrementally and generating speech output via a TTS module.

4. **Training stages:** 
   - Stage 1: Pretrain vision adapter (text+image pairs)
   - Stage 2: Add video understanding
   - Stage 3: Add audio understanding + generation
   - Stage 4: Full multimodal instruction tuning

This progressive training prevents catastrophic forgetting and allows each modality to bootstrap from the capabilities of previous stages.

## 8.2 What Makes MiniCPM-o Work at 8B

| Component | Model | Params | Role |
|-----------|-------|--------|------|
| Vision | SigLIP-400M | 400M | High-res image understanding |
| Audio | Whisper-large-v3 | ~600M | Speech recognition |
| LLM | Qwen2-7B | 7B | Reasoning backbone |
| TTS | ChatTTS | ~300M | Speech generation |
| **Total** | | **~8.3B** | |

**The lesson:** You don't need a 70B+ model for multimodal capability. By choosing strong, specialized encoders for each modality and connecting them through a capable but moderately-sized LLM, you can achieve competitive results.

**Comparison with Chameleon 34B:**
- Chameleon trains everything from scratch (one VQ tokenizer for images) — clean but expensive
- MiniCPM-o reuses pretrained specialists — practical and parameter-efficient
- MiniCPM-o achieves comparable or better results on understanding benchmarks despite being 4× smaller

# 9) Where the Field Is Heading

## 9.1 From Multimodal Models to World Models

The trajectory from Chameleon → Transfusion → Janus Pro → next-generation models points toward **world models** — systems that don't just process and generate modalities, but **understand the physics and causal structure** of the world.

```
Evolution:
  2023: "Glue" models        — understand images, generate text only
  2024: Native multimodal    — understand AND generate images + text
  2025: Any-to-any           — all modalities, streaming, real-time
  2026+: World models         — predict what happens next in the physical world
```

## 9.2 Key Open Problems

### The Tokenizer Bottleneck
Current VQ-VAE tokenizers lose information during quantization. The reconstruction quality of the tokenizer is a hard ceiling on generation quality. Research directions:
- **Finite Scalar Quantization (FSQ):** Replace VQ with simple scalar quantization — each dimension is independently quantized to a small set of values. No codebook collapse, simpler training.
- **Residual Quantization (RQ):** Multiple VQ passes, each quantizing the residual from the previous pass. Exponentially larger effective codebook.
- **Continuous tokenizers (Transfusion approach):** Avoid discretization entirely for generation.

### The Sequence Length Problem
A single 1024×1024 image at 16× downsampling = 4096 tokens. A 10-second video at 24fps = ~100K tokens. Current solutions:
- **Token merging/pruning** during attention
- **Perceiver-style** cross-attention bottlenecks
- **State-space models** (Mamba) replacing attention for long sequences

### Real-Time Generation
Current models generate images token-by-token (slow) or via diffusion (parallel but requires many denoising steps). The goal:
- **Consistency models:** Single-step generation
- **Speculative decoding** for parallel image token generation
- **Streaming generation:** Start rendering before all tokens are generated

### Embodied Agents
The ultimate test: can these models control robots?
- **Input:** Camera feed (video), proprioception (sensor data), language instructions
- **Output:** Motor commands (continuous actions), verbal responses
- All through a single multimodal model

## 9.3 The Convergence Thesis

The field appears to be converging on a key insight: **the right architecture is not one-size-fits-all.** Janus Pro's success with decoupled encoders suggests that future models will likely have:

1. **Shared reasoning backbone** — a large transformer that handles all cross-modal reasoning
2. **Specialized input encoders** — optimized for each modality's understanding needs
3. **Specialized output decoders** — optimized for each modality's generation needs
4. **Unified training** — all components trained end-to-end on interleaved data

This is not so different from how the human brain works: shared cortical circuits for reasoning, with specialized sensory cortices for each modality.

---

## Summary

| Concept | Key Takeaway |
|---------|-------------|
| VQ-VAE | Bridges continuous signals ↔ discrete tokens via learned codebooks |
| Unified vocabulary | Text + image + audio tokens in one flat vocabulary; one embedding table, one softmax |
| Interleaved data | Natural co-occurrence of modalities teaches cross-modal reasoning |
| Pure AR (Chameleon) | Simplest architecture; one loss for all; but forces sequential image generation |
| Transfusion | AR for text + diffusion for images; best of both worlds but more complex |
| Janus Pro | Decoupled encoders for understanding vs generation; resolves the representation conflict |
| Scaling | Specialized encoders + moderate LLM (8B) can match larger monolithic models |